# 05 — Executive Dashboard: SaaS Product & Customer Analytics
Renders an in-notebook HTML dashboard using `displayHTML()` with KPI cards and summary tables.

In [ ]:
from pyspark.sql import functions as F


In [ ]:
# ── Pull KPIs ────────────────────────────────────────────────────────────────
kpi = spark.sql("SELECT * FROM vw_executive_summary").collect()[0]

total_accounts = f"{kpi['total_accounts']:,}"
total_mrr      = f"${kpi['total_mrr_usd']:,}"
avg_health     = f"{kpi['avg_health_score']:.1f}"
churn_rate     = f"{kpi['churn_rate_pct']:.1f}%"
at_risk        = f"{kpi['at_risk_accounts']:,}"

# Overall SLA breach rate
sla = spark.table("gold_support_performance") \
    .agg(F.round(F.sum("sla_breaches") * 100.0 / F.sum("total_tickets"), 1).alias("r")).collect()[0]["r"]
sla_breach_rate = f"{sla:.1f}%"

print("KPIs collected.")


In [ ]:
# ── Churn risk table ─────────────────────────────────────────────────────────
risk_rows = (
    spark.sql("SELECT * FROM vw_churn_risk_accounts")
    .limit(8).collect()
)

risk_html = ""
for r in risk_rows:
    h = r["health_score"]
    hc = "#d32f2f" if h < 30 else "#f57c00" if h < 45 else "#388e3c"
    risk_html += f"""<tr>
        <td>{r['account_id']}</td>
        <td>{r['plan']}</td>
        <td>${r['mrr_usd']:,}</td>
        <td>{r['region']}</td>
        <td style='color:{hc};font-weight:bold'>{h:.1f}</td>
        <td>{r['days_as_customer']} days</td>
    </tr>"""


In [ ]:
# ── Feature adoption table ───────────────────────────────────────────────────
feat_rows = (
    spark.sql("SELECT * FROM vw_feature_usage")
    .limit(8).collect()
)

feat_html = ""
for r in feat_rows:
    feat_html += f"""<tr>
        <td>{r['feature']}</td>
        <td>{r['feature_category']}</td>
        <td>{r['total_events']:,}</td>
        <td>{r['unique_users']:,}</td>
        <td>{r['unique_accounts']:,}</td>
        <td>{r['avg_duration_secs']:.0f}s</td>
    </tr>"""


In [ ]:
# ── Render Dashboard ─────────────────────────────────────────────────────────
html = f"""
<style>
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f5f5f5;padding:24px;}}
  h1{{color:#1565c0;font-size:1.6em;margin-bottom:4px;}}
  .subtitle{{color:#666;margin-bottom:20px;font-size:.9em;}}
  .kpi-grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin-bottom:28px;}}
  .kpi{{background:#fff;border-radius:10px;padding:18px 20px;box-shadow:0 2px 8px rgba(0,0,0,.07);}}
  .kpi .label{{font-size:.78em;color:#888;text-transform:uppercase;letter-spacing:.05em;}}
  .kpi .value{{font-size:1.9em;font-weight:700;color:#1565c0;margin-top:4px;}}
  .section{{background:#fff;border-radius:10px;padding:20px;box-shadow:0 2px 8px rgba(0,0,0,.07);margin-bottom:24px;}}
  .section h3{{margin:0 0 14px;color:#333;font-size:1em;}}
  table{{width:100%;border-collapse:collapse;font-size:.88em;}}
  th{{background:#1565c0;color:#fff;padding:8px 12px;text-align:left;font-weight:600;}}
  td{{padding:7px 12px;border-bottom:1px solid #eee;}}
  tr:hover td{{background:#f0f4ff;}}
</style>
<h1>💻 SaaS Product & Customer Analytics</h1>
<div class='subtitle'>Technology & Software | Live Account Portfolio</div>

<div class='kpi-grid'>
  <div class='kpi'><div class='label'>Total Accounts</div><div class='value'>{total_accounts}</div></div>
  <div class='kpi'><div class='label'>Monthly Recurring Revenue</div><div class='value' style='color:#388e3c'>{total_mrr}</div></div>
  <div class='kpi'><div class='label'>Churn Rate</div><div class='value' style='color:#d32f2f'>{churn_rate}</div></div>
  <div class='kpi'><div class='label'>Avg Health Score</div><div class='value'>{avg_health}</div></div>
  <div class='kpi'><div class='label'>SLA Breach Rate</div><div class='value' style='color:#f57c00'>{sla_breach_rate}</div></div>
  <div class='kpi'><div class='label'>At-Risk Accounts</div><div class='value' style='color:#d32f2f'>{at_risk}</div></div>
</div>

<div class='section'>
  <h3>⚠️ Churn Risk Accounts (top by MRR)</h3>
  <table>
    <tr><th>Account</th><th>Plan</th><th>MRR</th><th>Region</th><th>Health Score</th><th>Tenure</th></tr>
    {risk_html}
  </table>
</div>

<div class='section'>
  <h3>📊 Top Features by Usage</h3>
  <table>
    <tr><th>Feature</th><th>Category</th><th>Events</th><th>Users</th><th>Accounts</th><th>Avg Duration</th></tr>
    {feat_html}
  </table>
</div>
"""

displayHTML(html)